In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv('../datasets/WineQT.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


Вообще, какое-то подобие EDA я делал в ноутбуке 02_decision_wine.ipynb, поэтому тут я его повторять не буду


In [3]:
X = df.drop(columns=['quality', 'Id'])
y = df['quality']

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3)
model = DecisionTreeClassifier(max_depth=8, criterion="gini", splitter="random")#почему-то конкретно здесь gini критернион оказался лучше остальных, оч странно
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.5626822157434402


In [17]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.6763848396501457


In [25]:
for n in [10, 25, 50, 100, 200, 300, 500, 1000]:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    train_y_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, train_y_pred)
    test_accuracy = accuracy_score(y_test, y_pred)
    print(f'n_estimators={n}', f'test_accuracy={test_accuracy}', f'train_accuracy={train_accuracy}')


n_estimators=10 test_accuracy=0.6355685131195336 train_accuracy=0.98375
n_estimators=25 test_accuracy=0.6763848396501457 train_accuracy=1.0
n_estimators=50 test_accuracy=0.6647230320699709 train_accuracy=1.0
n_estimators=100 test_accuracy=0.6530612244897959 train_accuracy=1.0
n_estimators=200 test_accuracy=0.673469387755102 train_accuracy=1.0
n_estimators=300 test_accuracy=0.6588921282798834 train_accuracy=1.0
n_estimators=500 test_accuracy=0.6705539358600583 train_accuracy=1.0
n_estimators=1000 test_accuracy=0.6705539358600583 train_accuracy=1.0


Randon Forest тоже может переобучиться, но куда тяжелее, чем одно дерево. Количество деревьев тоже влияет на переобучение. При подборе максимальной глубины далее возьмём n_estimators=100

In [24]:
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 30, 50, 100, 200, 250]:
    model = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    train_y_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, train_y_pred)
    test_accuracy = accuracy_score(y_test, y_pred)
    print(f'max_depth={depth}', f'test_accuracy={test_accuracy}', f'train_accuracy={train_accuracy}')

max_depth=1 test_accuracy=0.5626822157434402 train_accuracy=0.5725
max_depth=2 test_accuracy=0.5889212827988338 train_accuracy=0.60125
max_depth=3 test_accuracy=0.6064139941690962 train_accuracy=0.62
max_depth=4 test_accuracy=0.6034985422740525 train_accuracy=0.67
max_depth=5 test_accuracy=0.60932944606414 train_accuracy=0.715
max_depth=6 test_accuracy=0.6326530612244898 train_accuracy=0.7725
max_depth=8 test_accuracy=0.6443148688046647 train_accuracy=0.89875
max_depth=10 test_accuracy=0.6647230320699709 train_accuracy=0.9825
max_depth=15 test_accuracy=0.641399416909621 train_accuracy=1.0
max_depth=20 test_accuracy=0.6501457725947521 train_accuracy=1.0
max_depth=30 test_accuracy=0.6530612244897959 train_accuracy=1.0
max_depth=50 test_accuracy=0.6530612244897959 train_accuracy=1.0
max_depth=100 test_accuracy=0.6530612244897959 train_accuracy=1.0
max_depth=200 test_accuracy=0.6530612244897959 train_accuracy=1.0
max_depth=250 test_accuracy=0.6530612244897959 train_accuracy=1.0


Казалось бы, если бы мы взяли такую высокую глубину дерева на обычном дереве решений, мы бы получили дикое переобучение, да? Но здесь переобуение не так ярко выражено, после max_depth=30 тест аккураси почему-то замер


- Random Forest показал accuracy выше Decision Tree.
- Увеличение числа деревьев после 200 перестало улучшать качество.
- Глубина больше 10 уже почти не помогает, но при этом метрика не сильно просаживается.
- Лес сильно менее склонен к переобучению, чем одиночное дерево.

(у рандомного леса куда больше гиперпараметров, поэтому целесообразно потыкать их)